# E15 — O mundo que não foi observado

O andar começa pela pergunta mais dura do livro: de que serve uma proteção calibrada no que já se
viu, quando o prejuízo é decidido pelo mundo que **não** se viu?

A resposta natural — dimensionar pelo pior dia do registro — tem duas falhas, e as duas se medem.
A primeira é que o pior dia do registro não é um teto: há uma probabilidade exata de que um dia
futuro passe por cima dele. A segunda é que o recorde não cai por pouco: quando cai, ele passa
longe do segundo lugar.

A conta exata é a mesma estrutura do primeiro capítulo. Se os dias observados e os dias por vir
fossem permutáveis, o maior valor do conjunto estaria em qualquer posição com a mesma chance, e a
probabilidade de que algum dos próximos m dias supere o recorde dos n já vistos é m dividido por
n mais m. Com um ano de história e um ano por vir, isso é meio.


In [1]:
# <- brinque com: SERIES, FAT0RES, SEMENTES, DIAS_POR_MUNDO, QUANDO
import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import frevolab
from frevolab import dados, graficos, mudanca, recorde, volatilidade

SERIES = ("sp500.csv", "ibov.csv", "btc.csv")
FATORES = (1.0, 1.2, 1.5, 2.0, 3.0)
SEMENTES = 40
DIAS_POR_MUNDO = 12000
QUANDO = 6000
SEMENTE = 700
SORTEIOS = 4000

print("a conta: um ano de historia contra um ano por vir da %.3f" % recorde.probabilidade(252, 252))
print("o proximo dia contra um registro de %d dias: %.5f" % (6718, recorde.do_proximo(6718)))
print("recordes esperados num mundo sem mudanca: %d dias dao %.2f | %d dias dao %.2f"
      % (DIAS_POR_MUNDO, recorde.esperado(DIAS_POR_MUNDO), 1000, recorde.esperado(1000)))


a conta: um ano de historia contra um ano por vir da 0.500
o proximo dia contra um registro de 6718 dias: 0.00015
recordes esperados num mundo sem mudanca: 12000 dias dao 9.97 | 1000 dias dao 7.49


In [2]:
# O dado real: os recordes, e o tamanho do salto do ultimo.
linhas = []
for arquivo in SERIES:
    perdas = -volatilidade.retornos_log(dados.carregar_serie(arquivo)).to_numpy()
    dois = np.sort(perdas)[-2:]
    linhas.append({"serie": arquivo.replace(".csv", ""), "dias": perdas.size,
                   "recordes": recorde.conta(perdas), "esperado": recorde.esperado(perdas.size),
                   "pior": float(dois[1]), "segundo": float(dois[0]),
                   "razao": float(dois[1] / dois[0])})
reais = pd.DataFrame(linhas).set_index("serie")
print(reais.round(4).to_string())
print()
print("os recordes medidos ficam abaixo da conta em todas as tres: %s"
      % {s: (int(reais.loc[s, "recordes"]), round(reais.loc[s, "esperado"], 1)) for s in reais.index})


       dias  recordes  esperado    pior  segundo   razao
serie                                                   
sp500  6718         6    9.3898  0.1277   0.0999  1.2772
ibov   6620         8    9.3751  0.1599   0.1499  1.0668
btc    4387         6    8.9637  0.4647   0.2376  1.9563

os recordes medidos ficam abaixo da conta em todas as tres: {'sp500': (6, np.float64(9.4)), 'ibov': (8, np.float64(9.4)), 'btc': (6, np.float64(9.0))}


In [3]:
# A conta conferida por sorteio, e o mundo sem mudanca.
conferencia = []
for n, m in ((60, 60), (252, 252), (1260, 252)):
    venceu = 0
    for s in range(SORTEIOS):
        sorteio = np.random.default_rng(2000 + s)
        if sorteio.normal(0, 1, m).max() > sorteio.normal(0, 1, n).max():
            venceu += 1
    conferencia.append({"n": n, "m": m, "previsto": recorde.probabilidade(n, m),
                        "medido": venceu / SORTEIOS})
tabela_conferencia = pd.DataFrame(conferencia).set_index(["n", "m"])
print(tabela_conferencia.round(4).to_string())
parado = []
for s in range(SEMENTES):
    x = np.abs(mudanca.degrau(DIAS_POR_MUNDO, np.random.default_rng(SEMENTE + s), fator=1.0,
                              quando=QUANDO))
    parado.append(recorde.conta(x))
parado = np.array(parado)
print()
print("mundo sem mudanca: recordes %.2f (dp %.2f) | a conta diz %.2f"
      % (parado.mean(), parado.std(ddof=1), recorde.esperado(DIAS_POR_MUNDO)))


          previsto  medido
n    m                    
60   60     0.5000  0.5038
252  252    0.5000  0.4972
1260 252    0.1667  0.1592

mundo sem mudanca: recordes 9.88 (dp 2.88) | a conta diz 9.97


In [4]:
# O mundo que muda: recordes, e os dias que passam por cima do extremo do passado.
linhas = []
for fator in FATORES:
    contagens, fracoes, razoes = [], [], []
    for s in range(SEMENTES):
        x = np.abs(mudanca.degrau(DIAS_POR_MUNDO, np.random.default_rng(SEMENTE + s), fator=fator,
                                  quando=QUANDO))
        contagens.append(recorde.conta(x))
        f = recorde.faltou(x[:QUANDO], x[QUANDO:])
        fracoes.append(f["fracao"])
        razoes.append(f["razao"])
    linhas.append({"fator": fator, "recordes": float(np.mean(contagens)),
                   "recordes_dp": float(np.std(contagens, ddof=1)),
                   "fracao_alem": float(np.mean(fracoes)),
                   "razao_dos_extremos": float(np.mean(razoes))})
mudados = pd.DataFrame(linhas).set_index("fator")
print(mudados.round(4).to_string())
print()
print("a conta dos recordes e sempre %.2f, e o mundo que dobra faz %.2f"
      % (recorde.esperado(DIAS_POR_MUNDO), mudados.loc[2.0, "recordes"]))


       recordes  recordes_dp  fracao_alem  razao_dos_extremos
fator                                                        
1.0       9.875       2.8840       0.0002              0.9957
1.2      11.375       2.5185       0.0015              1.1948
1.5      13.675       2.9733       0.0107              1.4935
2.0      15.350       3.1586       0.0541              1.9913
3.0      16.725       3.4640       0.1959              2.9870

a conta dos recordes e sempre 9.97, e o mundo que dobra faz 15.35


In [5]:
# Figura 1: a escada do recorde, no indice.
perdas = -volatilidade.retornos_log(dados.carregar_serie(SERIES[0])).to_numpy()
marca = recorde.recordes(perdas)
escada = np.maximum.accumulate(perdas)
fig, eixo = plt.subplots(figsize=(8.6, 4.4))
eixo.plot(np.arange(perdas.size), escada, color="#1f4e79", lw=1.6, label="o recorde ate aqui")
eixo.scatter(np.arange(perdas.size)[marca], perdas[marca], s=42, color="#b03a2e", zorder=5,
             label="o dia que fez o recorde")
eixo.set_xlabel("dia da serie")
eixo.set_ylabel("pior perda ate aqui")
eixo.legend(frameon=False, fontsize=9)
eixo.grid(alpha=0.25)
fig.tight_layout()
graficos.salvar(fig, "E15_mundo_que_faltou", 1)
plt.close(fig)
print("recordes do indice: %d | o ultimo em %d dias de serie"
      % (int(marca.sum()), int(np.flatnonzero(marca)[-1])))


recordes do indice: 6 | o ultimo em 5080 dias de serie


In [6]:
# Figura 2: o que passa por cima do extremo do passado, contra o tamanho da mudanca.
fig, eixo = plt.subplots(figsize=(8.6, 4.4))
eixo.plot(mudados.index, mudados["fracao_alem"], marker="o", color="#1f4e79", lw=1.6,
          label="dias do futuro acima do extremo do passado")
eixo.axhline(0.0, color="#555555", ls=":", lw=1.2, label="mundo sem mudanca")
for fator, rotulo, cor in ((1.0, "sem mudanca", "#7f8c8d"), (2.0, "dobra", "#b03a2e")):
    if fator in mudados.index:
        eixo.annotate("%.3f" % mudados.loc[fator, "fracao_alem"], (fator, mudados.loc[fator, "fracao_alem"]),
                      textcoords="offset points", xytext=(6, 6), fontsize=9, color=cor)
eixo.set_xlabel("fator da mudanca no meio da serie")
eixo.set_ylabel("fracao dos dias que passam por cima do recorde")
eixo.legend(frameon=False, fontsize=9)
eixo.grid(alpha=0.25)
fig.tight_layout()
graficos.salvar(fig, "E15_mundo_que_faltou", 2)
plt.close(fig)
print("fracoes: %s" % {f: round(v, 4) for f, v in mudados["fracao_alem"].items()})


fracoes: {1.0: 0.0002, 1.2: 0.0015, 1.5: 0.0107, 2.0: 0.0541, 3.0: 0.1959}


## Leitura visual das figuras

A preencher olhando os .png com a ponte de visão (AGENTS.md §9). Observação, não número.


In [7]:
# O resultado: um objeto por grandeza, para o livro citar por comando.
NOMES = {1.0: "sem_mudanca", 1.2: "vinte_por_cento", 1.5: "metade", 2.0: "dobro", 3.0: "triplo"}
resultado = {
    "recorde_probabilidade": float(recorde.probabilidade(252, 252)),
    "recorde_do_proximo": float(recorde.do_proximo(6718)),
    "recorde_esperado_mil": float(recorde.esperado(1000)),
    "recorde_esperado_mundo": float(recorde.esperado(DIAS_POR_MUNDO)),
    "recorde_sementes": int(SEMENTES),
    "recorde_dias_por_mundo": int(DIAS_POR_MUNDO),
    "recorde_sorteios": int(SORTEIOS),
    "recorde_medido_sessenta": float(tabela_conferencia.loc[(60, 60), "medido"]),
    "recorde_medido_ano": float(tabela_conferencia.loc[(252, 252), "medido"]),
    "recorde_parado": float(parado.mean()),
    "recorde_parado_dispersao": float(parado.std(ddof=1)),
}
for arquivo, curto in zip(SERIES, ("indice", "ibovespa", "bitcoin")):
    nome = arquivo.replace(".csv", "")
    resultado["recorde_%s_dias" % curto] = int(reais.loc[nome, "dias"])
    resultado["recorde_%s_conta" % curto] = int(reais.loc[nome, "recordes"])
    resultado["recorde_%s_esperado" % curto] = float(reais.loc[nome, "esperado"])
    resultado["recorde_%s_pior" % curto] = float(reais.loc[nome, "pior"])
    resultado["recorde_%s_segundo" % curto] = float(reais.loc[nome, "segundo"])
    resultado["recorde_%s_razao" % curto] = float(reais.loc[nome, "razao"])
for fator in FATORES:
    nome = NOMES[fator]
    resultado["recorde_recordes_%s" % nome] = float(mudados.loc[fator, "recordes"])
    resultado["recorde_fracao_%s" % nome] = float(mudados.loc[fator, "fracao_alem"])
    resultado["recorde_razao_%s" % nome] = float(mudados.loc[fator, "razao_dos_extremos"])
caminho = Path("lab/resultados/E15_mundo_que_faltou.json")
caminho.write_text(json.dumps(resultado, indent=1, ensure_ascii=False, sort_keys=True), encoding="utf-8")
print("%s gravado | %d grandezas" % (caminho, len(resultado)))


lab/resultados/E15_mundo_que_faltou.json gravado | 44 grandezas
